In [3]:
!java -version 2>&1 | head -1
!nproc
!free -h | head -2
!df -h /kaggle/working | tail -1

openjdk version "21.0.12" 2026-07-21
4
               total        used        free      shared  buff/cache   available
Mem:            31Gi       1.2Gi        25Gi       1.2Mi       5.2Gi        30Gi
/dev/loop2       20G   92K   20G   1% /kaggle/working


In [4]:
!pip uninstall -y dataproc-spark-connect

Found existing installation: dataproc-spark-connect 1.1.0
Uninstalling dataproc-spark-connect-1.1.0:
  Successfully uninstalled dataproc-spark-connect-1.1.0


In [5]:
!pip install -q pyspark==3.5.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 5.4 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 6.4 MB/s eta 0:00:00


In [6]:
import shutil
from pyspark.sql import SparkSession

def get_spark(app_name="cipherstop", driver_mem="8g"):   # adapt to the RAM from cell 1
    return (SparkSession.builder.master("local[*]").appName(app_name)
            .config("spark.driver.memory", driver_mem)
            .config("spark.sql.shuffle.partitions", "16")
            .config("spark.sql.session.timeZone", "UTC")
            .getOrCreate())

spark = get_spark()
assert spark.range(1_000_000).selectExpr("sum(id) as s").first()["s"] == 499999500000
spark.range(1000).write.mode("overwrite").parquet("/kaggle/working/_smoke")
assert spark.read.parquet("/kaggle/working/_smoke").count() == 1000
shutil.rmtree("/kaggle/working/_smoke")
print("Spark", spark.version, "OK")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/14 10:20:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/14 10:20:12 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/10/14 10:20:13 WARN DataSource: All paths were ignored:                      
  file:/kaggle/working/_smoke


Spark 3.5.3 OK


In [7]:
import os, re, collections

RAW_DIR = "/kaggle/input/datasets/hiranomanabu/ransmap-2024-ransomware-behavioral-features/RanSMAP/dataset"
STREAMS = ["ata_read", "ata_write", "mem_exec", "mem_readwrite", "mem_read", "mem_write"]
NAME_RE = re.compile(r"^(.+)-(\d{8}_\d{2}-\d{2}-\d{2})$")   # Conti_03-20240227_21-16-45

execs = {}   # rel_dir -> {stream: size_in_bytes}
for root, _, files in os.walk(RAW_DIR):
    csvs = [f for f in files if f.endswith(".csv")]
    if csvs:
        rel = os.path.relpath(root, RAW_DIR).replace(os.sep, "/")
        execs[rel] = {f[:-4]: os.path.getsize(os.path.join(root, f)) for f in csvs}

def parse(rel):
    leaf = rel.split("/")[-1]
    mt = NAME_RE.match(leaf)
    sample = mt.group(1) if mt else leaf                 # Conti_03
    family = re.sub(r"[_-]?\d+$", "", sample)            # Conti
    return sample, family

print(len(execs), "executions | total GB:",
      round(sum(sum(v.values()) for v in execs.values()) / 1e9, 1))

1970 executions | total GB: 172.1


In [8]:
one = sorted(execs)[0]
print(one)
for s in STREAMS:
    p = f"{RAW_DIR}/{one}/{s}.csv"
    if os.path.exists(p):
        with open(p) as fh:
            head = [fh.readline().rstrip() for _ in range(3)]
            n = 3 + sum(1 for _ in fh)
        print(f"== {s}: {os.path.getsize(p)} bytes, {n} lines")
        print("\n".join(head))
    else:
        print(f"== {s}: missing")

for s in STREAMS:
    print(s, "non-empty in", sum(1 for v in execs.values() if v.get(s, 0) > 0), "executions")

extra/i3-gen12/ddr5-4800-16g/AESCrypt/AESCrypt-20240228_10-57-21
== ata_read: 30499109 bytes, 739693 lines
1709085154, 410817795,7722400,4096,-1,0
1709085154, 411883781,38952104,4096,-1,0
1709085154, 411884854,38952112,4096,-1,0
== ata_write: 41317478 bytes, 717724 lines
1709085154, 846308834,7407088,4096,0.11740315732352907,0
1709085154, 941840673,7714824,4096,0.27670045020940726,0
1709085155,   1625073,7714912,4096,0.3193659678695793,0
== mem_exec: 14449 bytes, 357 lines
1709085153, 544244470,6010152281,0,-1,2
1709085153, 544311027,6464946212,0,-1,2
1709085153, 544320689,5985759702,0,-1,2
== mem_readwrite: 55318 bytes, 932 lines
1709085153, 549609823,71498384,4096,0.6264335986898634,2
1709085153, 550364494,6410224864,4096,0.7026246815529066,2
1709085153, 551034358,18842032,4096,0.292475160486166,2
== mem_read: 11734441 bytes, 293257 lines
1709085153, 544194016,5438812160,0,-1,2
1709085153, 544214412,6644719648,0,-1,2
1709085153, 544224074,6534631440,0,-1,2
== mem_write: 12819581 byte

In [9]:
import pandas as pd

RANSOM = {"Conti", "LockBit", "Darkside", "Ryuk", "REvil", "WannaCry"}
BENIGN = {"Idle", "Firefox", "SDelete", "Zip", "AESCrypt", "Office"}
ZERODAY_FAMILY = "Ryuk"

# hardware configs held out of training (team decision, see the notes above)
TEST_HW = {"celeron-gen6/ddr4-1866-8g", "i3-gen12/ddr4-3200-16g", "i5-gen12/ddr4-2133-16g"}
VAL_HW  = {"celeron-gen6/ddr4-2133-16g", "i3-gen12/ddr4-2666-8g"}
MANIFEST = "/kaggle/working/manifest.csv"

rows = []
for rel, files in execs.items():
    parts = rel.split("/")                    # [top, cpu, ram, activity, execution]
    sample, family = parse(rel)
    comp = set(family.split("_"))
    assert comp <= RANSOM | BENIGN, (rel, comp - RANSOM - BENIGN)
    rw = sorted(comp & RANSOM)
    assert len(rw) <= 1, rel
    rows.append({"rel_dir": rel, "sample_id": rel.replace("/", "__"),
                 "top": parts[0], "hw": f"{parts[1]}/{parts[2]}",
                 "sample": sample, "family": family,
                 "ransomware": rw[0] if rw else "none",
                 "mixed": int(len(comp) > 1), "label": int(bool(rw)),
                 "has_ata": int("ata_read" in files and "ata_write" in files)})
m = pd.DataFrame(rows)
assert (TEST_HW | VAL_HW) <= set(m.hw), "unknown hardware name"

def split_of(r):
    if r.ransomware == ZERODAY_FAMILY: return "zeroday"
    if r.top == "mix":      return "mix"          # evaluation only
    if r.top == "variants": return "variants"     # evaluation only
    if r.hw in TEST_HW:     return "test"
    if r.hw in VAL_HW:      return "val"
    return "train"
m["split"] = m.apply(split_of, axis=1)
m.to_csv(MANIFEST, index=False)

print(m.groupby(["split", "label"]).size().unstack(fill_value=0))
print(m.groupby(["split", "family"]).size().unstack(fill_value=0).T)

main = m[m.split.isin(["train", "val", "test"])]
assert main.groupby("hw").split.nunique().max() == 1, "a hardware config is in two splits"
assert main.groupby("split").label.nunique().min() == 2, "a split is missing a class"
assert not ((m.ransomware == ZERODAY_FAMILY) & (m.split != "zeroday")).any(), "zero-day leak"
print("excluded (missing ata file):", (m.has_ata == 0).sum(), "| unique ids:", m.sample_id.is_unique)

label       0    1
split             
mix         0  100
test      180  150
train     600  500
val       120  100
variants    0   70
zeroday     0  150
split                 mix  test  train  val  variants  zeroday
family                                                        
AESCrypt                0    30    100   20         0        0
AESCrypt_Conti         10     0      0    0         0        0
AESCrypt_REvil         10     0      0    0         0        0
Conti                   0    30    100   20        70        0
Darkside                0    30    100   20         0        0
Firefox                 0    30    100   20         0        0
Firefox_Conti          10     0      0    0         0        0
Firefox_Office_REvil   10     0      0    0         0        0
Firefox_REvil          10     0      0    0         0        0
Idle                    0    30    100   20         0        0
LockBit                 0    30    100   20         0        0
Office                  0    

In [10]:
import pandas as pd
from pyspark.sql import functions as F

ALL_STREAMS = ["ata_read", "ata_write", "mem_exec", "mem_readwrite", "mem_read", "mem_write"]
EVENT_DDL = "sec long, ns double, lba long, size long, entropy1 double, c6 double"

one_per_family = m[m.has_ata == 1].groupby("family").head(1)
res = []
for s in ALL_STREAMS:
    paths = [f"{RAW_DIR}/{r}/{s}.csv" for r in one_per_family.rel_dir]
    raw = spark.read.option("header", False).schema(EVENT_DDL).csv(paths)
    r_ = raw.agg(
        F.count("*").alias("rows"),
        F.sum((F.col("sec").isNull() | F.col("ns").isNull() | F.col("lba").isNull()
               | F.col("size").isNull() | F.col("entropy1").isNull()).cast("int")).alias("rows_with_null"),
        F.sum((F.col("size") < 0).cast("int")).alias("neg_size"),
        F.sum(((F.col("ns") < 0) | (F.col("ns") >= 1e9)).cast("int")).alias("ns_out_of_range"),
        F.sum((F.col("entropy1") == -1).cast("int")).alias("entropy_is_minus1"),
        F.sum(((F.col("entropy1") < 0) & (F.col("entropy1") != -1)).cast("int")).alias("entropy_other_neg"),
        F.sum((F.col("entropy1") > 1).cast("int")).alias("entropy_gt1"),
        F.min("size").alias("size_min"), F.max("size").alias("size_max"),
        F.countDistinct("c6").alias("c6_distinct"), F.max("c6").alias("c6_max"),
    ).first().asDict()
    r_["stream"] = s
    res.append(r_)
pd.set_option("display.width", 250)
print(pd.DataFrame(res).set_index("stream").T)

stream               ata_read   ata_write  mem_exec  mem_readwrite   mem_read   mem_write
rows               11239219.0  16361652.0    5003.0         7395.0  9792374.0  10572967.0
rows_with_null            0.0         0.0       0.0            0.0        0.0         0.0
neg_size                  0.0         0.0       0.0            0.0        0.0         0.0
ns_out_of_range           0.0         0.0       0.0            0.0        0.0         0.0
entropy_is_minus1  11239219.0         0.0    5003.0            0.0  9792374.0         0.0
entropy_other_neg         0.0         0.0       0.0            0.0        0.0         0.0
entropy_gt1               0.0         0.0       0.0            0.0        0.0         0.0
size_min                 16.0        16.0       0.0         4096.0        0.0         0.0
size_max               4096.0      4096.0    4096.0         4096.0     4096.0      4096.0
c6_distinct               1.0         1.0       2.0            2.0        3.0         3.0
c6_max    

In [11]:
import re, time, os, json
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.window import Window

N_PER_FAMILY = 2          # None = full run
RUN = "subset" if N_PER_FAMILY else "full"
OUT_ROOT = f"/kaggle/working/interim_{RUN}"     # a subset can never be mistaken for the full run
WINDOW_S = 1.0
ACTIVE_STREAMS = ["ata_read", "ata_write", "mem_exec", "mem_readwrite", "mem_read", "mem_write"]
HAS_ENTROPY = {"ata_write", "mem_readwrite", "mem_write"}      # entropy = -1 in the other streams
EVENT_DDL = "sec long, ns double, addr long, size long, entropy1 double, c6 double"   # addr = LBA (ata) / GPA (mem)

pool = m[m.has_ata == 1]
sel = pool if N_PER_FAMILY is None else \
      pool.sample(frac=1, random_state=0).groupby("family").head(N_PER_FAMILY)
print(f"RUN={RUN} | executions selected: {len(sel)}")

report = {}

def ingest_stream(stream):
    if not os.path.exists(f"{OUT_ROOT}/{stream}/_SUCCESS"):
        paths = [f"{RAW_DIR}/{r}/{stream}.csv" for r in sel.rel_dir]
        root = re.escape(RAW_DIR.strip("/"))
        df = (spark.read.option("header", False).schema(EVENT_DDL).csv(paths)
              .withColumn("_p", F.expr("url_decode(input_file_name())"))
              .withColumn("rel_dir", F.regexp_extract("_p", rf"^file:/*{root}/(.*)/[^/]+$", 1))
              .drop("_p"))

        # validation rule: a row is valid if no field is null and size >= 0
        # (entropy = -1 is a valid "not available" marker; malformed rows parse as nulls)
        ok = (F.col("sec").isNotNull() & F.col("ns").isNotNull() & F.col("addr").isNotNull()
              & F.col("size").isNotNull() & (F.col("size") >= 0))
        t = F.col("sec") + F.col("ns") / 1e9
        ev = df.withColumn("ok", ok).withColumn("win", F.floor(t / WINDOW_S).cast("long"))
        v = lambda x: F.when(F.col("ok"), x)          # value counts only if the row is valid

        aggs = [F.sum(F.col("ok").cast("int")).alias(f"{stream}_n"),
                F.count("*").alias(f"{stream}_rawcnt"),
                F.sum(v(F.col("size"))).alias(f"{stream}_size_sum"),
                (F.max(v(F.col("addr"))) - F.min(v(F.col("addr")))).alias(f"{stream}_addr_range")]
        if stream in HAS_ENTROPY:
            e = F.when(F.col("ok") & (F.col("entropy1") >= 0), F.col("entropy1"))
            aggs += [F.mean(e).alias(f"{stream}_ent_mean"), F.max(e).alias(f"{stream}_ent_max")]

        ev.groupBy("rel_dir", "win").agg(*aggs).write.mode("overwrite").parquet(f"{OUT_ROOT}/{stream}")
    else:
        print(f"{stream}: already written, skipping")

    out = spark.read.parquet(f"{OUT_ROOT}/{stream}")
    s = out.agg(F.sum(f"{stream}_rawcnt").alias("raw"), F.sum(f"{stream}_n").alias("valid"),
                F.countDistinct("rel_dir").alias("executions"),
                F.sum(F.col("win").isNotNull().cast("int")).alias("windows")).first()
    report[stream] = {"raw_rows": s["raw"], "valid_rows": s["valid"],
                      "removed_rows": s["raw"] - s["valid"],
                      "executions": s["executions"], "windows": s["windows"]}
    print(f"{stream:14s} executions: {s['executions']} | windows: {s['windows']} | "
          f"rows: {s['raw']:,} | removed: {s['raw'] - s['valid']:,}")

t0 = time.time()
for s_ in ACTIVE_STREAMS:
    ingest_stream(s_)
elapsed = time.time() - t0

sub_gb = sum(execs[r].get(s, 0) for r in sel.rel_dir for s in ACTIVE_STREAMS) / 1e9
full_gb = sum(execs[r].get(s, 0) for r in pool.rel_dir for s in ACTIVE_STREAMS) / 1e9
print(f"elapsed: {elapsed:.0f} s for {sub_gb:.1f} GB -> full run ({full_gb:.0f} GB) roughly "
      f"{full_gb / sub_gb * elapsed / 3600:.1f} h if this run is representative")

with open(f"/kaggle/working/validation_report_{RUN}.json", "w") as fh:
    json.dump({"window_s": WINDOW_S, "executions": len(sel), "streams": report}, fh, indent=2)

RUN=subset | executions selected: 44


ata_read       executions: 44 | windows: 7641 | rows: 20,806,276 | removed: 0


ata_write      executions: 44 | windows: 7843 | rows: 31,316,121 | removed: 0
mem_exec       executions: 44 | windows: 739 | rows: 10,367 | removed: 0
mem_readwrite  executions: 44 | windows: 3053 | rows: 10,799 | removed: 0


mem_read       executions: 44 | windows: 8468 | rows: 19,785,959 | removed: 0


mem_write      executions: 44 | windows: 8503 | rows: 21,207,542 | removed: 0
elapsed: 239 s for 4.2 GB -> full run (172 GB) roughly 2.7 h if this run is representative


In [12]:
meta = (spark.read.option("header", True).csv(MANIFEST)
        .withColumn("label", F.col("label").cast("int"))
        .select("rel_dir", "sample_id", "top", "hw", "family", "ransomware", "mixed", "label", "split"))

dfs = [spark.read.parquet(f"{OUT_ROOT}/{s}").filter(F.col("win").isNotNull()).drop(f"{s}_rawcnt")
       for s in ACTIVE_STREAMS]
merged = reduce(lambda a, b: a.join(b, ["rel_dir", "win"], "full_outer"), dfs)

win = merged.join(F.broadcast(meta), "rel_dir", "inner")
win = (win.withColumn("step", F.col("win") - F.min("win").over(Window.partitionBy("rel_dir")))
          .drop("win"))        # step 0 = first window with any event in any stream

win.write.mode("overwrite").partitionBy("split").parquet(f"{OUT_ROOT}/windows")
print("rows:", spark.read.parquet(f"{OUT_ROOT}/windows").count())

26/10/14 10:25:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


rows: 8503


In [13]:
d = spark.read.parquet(f"{OUT_ROOT}/windows")

d.groupBy("split", "label").agg(F.countDistinct("sample_id").alias("executions"),
                                F.count("*").alias("windows")).orderBy("split", "label").show()

per = d.groupBy("sample_id").agg(F.count("*").alias("windows"), F.max("step").alias("max_step"))
per.select(F.min("windows"), F.avg("windows"), F.max("windows")).show()

n_gap  = per.filter(F.col("max_step") + 1 != F.col("windows")).count()
n_short = per.filter(F.col("windows") < 40).count()
n_neg = d.filter(" OR ".join(f"`{c}` < 0" for c in d.columns
                             if c.endswith(("_n", "_size_sum", "_addr_range")))).count()
zd_leak = d.filter((F.col("ransomware") == ZERODAY_FAMILY) & (F.col("split") != "zeroday")).count()
n_out = d.select("sample_id").distinct().count()

print("share of windows with NO events, per stream:")
d.select([F.avg(F.col(c).isNull().cast("int")).alias(c) for c in d.columns if c.endswith("_n")]).show(truncate=False)

print(f"executions in output: {n_out} (selected: {len(sel)})")
print(f"VALIDATION  gaps in step: {n_gap} | executions < 40 windows: {n_short} | "
      f"negative values: {n_neg} | zero-day leak: {zd_leak}")
assert n_out == len(sel) and n_gap == 0 and n_short == 0 and n_neg == 0 and zd_leak == 0, "validation failed"
print("ALL VALIDATION RULES PASSED")

+--------+-----+----------+-------+
|   split|label|executions|windows|
+--------+-----+----------+-------+
|     mix|    1|        20|   3781|
|    test|    0|         2|    373|
|   train|    0|         8|   1562|
|   train|    1|         8|   1628|
|     val|    0|         2|    390|
|     val|    1|         1|    194|
|variants|    1|         1|    187|
| zeroday|    1|         2|    388|
+--------+-----+----------+-------+

+------------+------------+------------+
|min(windows)|avg(windows)|max(windows)|
+------------+------------+------------+
|         182|      193.25|         225|
+------------+------------+------------+

share of windows with NO events, per stream:
+-------------------+-------------------+------------------+------------------+--------------------+-----------+
|ata_read_n         |ata_write_n        |mem_exec_n        |mem_readwrite_n   |mem_read_n          |mem_write_n|
+-------------------+-------------------+------------------+------------------+-----------